In [14]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import plotly.graph_objects as go

# Folder path with CSV files
folder_path = r"D:\Data\2025\August\21 Aug Tm UCP spectra dual beam Agniva\Bi UCP Red emission\23 august\clean Particle\975nm power variation"

# Define double exponential function
def double_exponential(x, a1, b1, a2, b2, c):
    return a1 * np.exp(b1 * x) + a2 * np.exp(b2 * x) + c

# Smart initial guess for curve fitting
def get_initial_guess(x, y):
    a1 = max(y) - min(y)
    a2 = a1 / 2
    b1 = -5    # fast decay (~0.2 ms)
    b2 = -1    # slow decay (~1 ms)
    c = min(y)
    return [a1, b1, a2, b2, c]

# Convert decay rate b to lifetime τ
def safe_lifetime(b): return -1 / b if b != 0 else np.nan

# Plotly template
fig_template = go.layout.Template()
fig_template.layout = {
    'template': 'simple_white+presentation',
    'autosize': False,
    'width': 800,
    'height': 600,
    'xaxis': {
        'title': 'Time (ms)',
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 2.0,
        'tickwidth': 2.0,
        'ticklen': 6,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white',
    },
    'yaxis': {
        'title': 'Intensity (a.u.)',
        'ticks': 'inside',
        'mirror': 'ticks',
        'linewidth': 2.0,
        'tickwidth': 2.0,
        'ticklen': 6,
        'showline': True,
        'showgrid': False,
        'zerolinecolor': 'white'
    },
    'font': {
        'family': 'mathjax',
        'size': 22,
    }
}

# Process all .csv files in folder
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        print(f"\n📄 Processing: {filename}")

        try:
            data = pd.read_csv(file_path)
        except Exception as e:
            print(f"❌ Failed to read {filename}: {e}")
            continue

        # Check and extract 'time' and 'intensity' columns
        columns = [col.lower() for col in data.columns]
        if 'time' not in columns or 'intensity' not in columns:
            print("⚠️ Skipped: Missing 'time' or 'intensity' column.")
            continue

        time = data.iloc[:, columns.index('time')].values
        intensity = data.iloc[:, columns.index('intensity')].values

        # Select fitting regions
        mask_25_50 = (time >= 25) & (time <= 50)
        mask_75_100 = (time >= 75) & (time <= 100)
        time1, intensity1 = time[mask_25_50], intensity[mask_25_50]
        time2, intensity2 = time[mask_75_100], intensity[mask_75_100]

        # Check if data is sufficient
        if len(time1) < 5 or len(time2) < 5:
            print("⚠️ Skipped: Not enough data in one of the fitting ranges.")
            continue

        # Perform double exponential fits
        try:
            popt1, _ = curve_fit(double_exponential, time1, intensity1, p0=get_initial_guess(time1, intensity1), maxfev=10000)
        except RuntimeError:
            popt1 = [np.nan] * 5
            print("⚠️ Fit failed for 25–50 ms.")

        try:
            popt2, _ = curve_fit(double_exponential, time2, intensity2, p0=get_initial_guess(time2, intensity2), maxfev=10000)
        except RuntimeError:
            popt2 = [np.nan] * 5
            print("⚠️ Fit failed for 75–100 ms.")

        # Plot the results
        fig = go.Figure(layout=fig_template.layout)
        fig.add_trace(go.Scatter(x=time, y=intensity, mode='lines', name='Original Data', line=dict(color='black')))

        if not any(np.isnan(popt1)):
            yfit1 = double_exponential(time1, *popt1)
            fig.add_trace(go.Scatter(x=time1, y=yfit1, mode='lines', name='Fit 25–50 ms', line=dict(dash='dash', color='red')))

        if not any(np.isnan(popt2)):
            yfit2 = double_exponential(time2, *popt2)
            fig.add_trace(go.Scatter(x=time2, y=yfit2, mode='lines', name='Fit 75–100 ms', line=dict(dash='dash', color='blue')))

        fig.update_layout(title=f'Fitting: {filename}')
        fig.show()

        # Calculate percentage intensity changes
        mean_0_25 = np.mean(intensity[(time >= 0) & (time <= 25)])
        mean_45_50 = np.mean(intensity[(time >= 45) & (time <= 50)])
        mean_50_75 = np.mean(intensity[(time >= 50) & (time <= 75)])
        mean_95_100 = np.mean(intensity[(time >= 95) & (time <= 100)])

        percent_change_1 = ((mean_45_50 - mean_0_25) / mean_0_25) * 100 if mean_0_25 else np.nan
        percent_change_2 = ((mean_95_100 - mean_50_75) / mean_50_75) * 100 if mean_50_75 else np.nan

        # Compute lifetimes
        tau1_fast = safe_lifetime(popt1[1])
        tau1_slow = safe_lifetime(popt1[3])
        tau2_fast = safe_lifetime(popt2[1])
        tau2_slow = safe_lifetime(popt2[3])

        # Print results
        print(f"📊 Intensity change (0–25 ms → 45–50 ms): {percent_change_1:.2f}%")
        print(f"📊 Intensity change (50–75 ms → 95–100 ms): {percent_change_2:.2f}%")
        print(f"⏱️  Tau_fast (25–50 ms): {tau1_fast * 1000:.2f} μs")
        print(f"⏱️  Tau_slow (25–50 ms): {tau1_slow * 1000:.2f} μs")
        print(f"⏱️  Tau_fast (75–100 ms): {tau2_fast * 1000:.2f} μs")
        print(f"⏱️  Tau_slow (75–100 ms): {tau2_slow * 1000:.2f} μs")